In [8]:
pip install geopandas shapely geopy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import geopandas as gpd  # کتابخانه‌ای برای خواندن و تحلیل داده‌های مکانی (Shapefile)
import numpy as np  # برای کار با آرایه‌ها و محاسبات عددی
from shapely.geometry import Point  # برای کار با نقاط هندسی
from scipy.spatial.distance import cdist  # برای محاسبه فاصله اقلیدسی بین دو مجموعه نقاط
import random  # برای تولید عدد تصادفی و انتخاب تصادفی

# --- خواندن فایل‌های shapefile ---
houses_path = r'C:\\Users\\lenovo\\Desktop\\data\\houses.shp'  # مسیر فایل خانه‌ها
schools_path = r'C:\\Users\\lenovo\\Desktop\\data\\schools.shp'  # مسیر فایل مدارس

houses_gdf = gpd.read_file(houses_path)  # خواندن فایل shapefile خانه‌ها و تبدیل به GeoDataFrame
schools_gdf = gpd.read_file(schools_path)  # خواندن فایل shapefile مدارس

houses_centroids = houses_gdf.geometry.centroid  # استخراج مرکز هندسی هر خانه (برای چندضلعی‌ها)
schools_centroids = schools_gdf.geometry.centroid  # استخراج مرکز هندسی هر مدرسه

houses_coords = np.array([(pt.x, pt.y) for pt in houses_centroids])  # استخراج مختصات X,Y از مراکز خانه‌ها
schools_coords = np.array([(pt.x, pt.y) for pt in schools_centroids])  # استخراج مختصات X,Y از مراکز مدارس

distance_matrix = cdist(houses_coords, schools_coords, metric='euclidean')  # ساخت ماتریس فاصله بین هر خانه و مدرسه

# --- پارامترهای الگوریتم ژنتیک ---
num_houses = distance_matrix.shape[0]  # تعداد کل خانه‌ها (تعداد ردیف‌ها در ماتریس فاصله)
num_schools = distance_matrix.shape[1]  # تعداد مدارس (تعداد ستون‌ها در ماتریس فاصله)
school_capacity = 38  # ظرفیت هر مدرسه (حداکثر ۳۸ خانه)
population_size = 50  # تعداد افراد (کروموزوم‌ها) در هر نسل
generations = 200  # تعداد نسل‌ها برای اجرای الگوریتم ژنتیک
mutation_rate = 0.02  # احتمال جهش در هر نسل (۲٪)

# --- تابع شایستگی: محاسبه مجموع فاصله‌ها + جریمه برای مدارس بیش از ظرفیت ---
def fitness(individual):
    total_distance = 0  # مجموع فاصله‌ها
    school_counts = [0] * num_schools  # شمارنده تعداد خانه‌های اختصاص یافته به هر مدرسه
    for house_idx, school_idx in enumerate(individual):  # پیمایش تمام خانه‌ها
        total_distance += distance_matrix[house_idx, school_idx]  # جمع کردن فاصله آن خانه تا مدرسه انتخابی
        school_counts[school_idx] += 1  # افزایش شمارنده برای آن مدرسه
    penalty = sum((max(0, count - school_capacity) ** 2) for count in school_counts)  # جریمه برای اضافه‌ظرفیت‌ها
    return total_distance + penalty * 10000  # مجموع فاصله به علاوه جریمه (با وزن بالا)

# --- تولید یک کروموزوم تصادفی با رعایت ظرفیت ---
def generate_individual():
    allocation = []  # لیست تخصیص خانه‌ها به مدارس
    counts = [0] * num_schools  # شمارش تعداد خانه‌های اختصاص‌یافته به هر مدرسه
    for _ in range(num_houses):
        choices = [i for i in range(num_schools) if counts[i] < school_capacity]  # فقط مدارسی که ظرفیت دارند
        school = random.choice(choices)  # انتخاب تصادفی از بین مدارس مجاز
        allocation.append(school)  # اختصاص مدرسه به خانه
        counts[school] += 1  # افزایش شمارش مدرسه انتخاب‌شده
    return allocation

# --- انتخاب والدین با روش تورنومنت (Tournament Selection) ---
def selection(population):
    tournament = random.sample(population, 5)  # انتخاب تصادفی ۵ کروموزوم
    return min(tournament, key=fitness)  # برگرداندن بهترین کروموزوم از لحاظ شایستگی

# --- کراس‌اور تک‌نقطه‌ای برای ترکیب دو والد ---
def crossover(parent1, parent2):
    point = random.randint(1, num_houses - 2)  # انتخاب نقطه تصادفی برای تقسیم
    child = parent1[:point] + parent2[point:]  # ترکیب ژن‌ها تا نقطه مشخص‌شده
    return child

# --- جهش تصادفی: تغییر مدرسه اختصاص‌داده‌شده به یک خانه ---
def mutate(individual):
    if random.random() < mutation_rate:  # با احتمال جهش
        idx = random.randint(0, num_houses - 1)  # انتخاب خانه تصادفی
        individual[idx] = random.randint(0, num_schools - 1)  # اختصاص مدرسه جدید تصادفی
    return individual

# --- اجرای الگوریتم ژنتیک ---
population = [generate_individual() for _ in range(population_size)]  # تولید جمعیت اولیه
best_solution = None  # متغیر برای نگهداری بهترین جواب تا کنون
best_fitness = float('inf')  # مقدار شایستگی اولیه (بسیار زیاد)

for generation in range(generations):  # تکرار برای تعداد نسل‌ها
    new_population = []
    for _ in range(population_size):
        parent1 = selection(population)  # انتخاب والد اول
        parent2 = selection(population)  # انتخاب والد دوم
        child = crossover(parent1, parent2)  # تولید فرزند با کراس‌اور
        child = mutate(child)  # انجام جهش روی فرزند
        new_population.append(child)  # افزودن فرزند به جمعیت جدید
    population = new_population  # جایگزینی جمعیت قبلی
    gen_best = min(population, key=fitness)  # بهترین کروموزوم نسل جاری
    gen_best_fit = fitness(gen_best)
    if gen_best_fit < best_fitness:  # به‌روزرسانی بهترین جواب تا کنون
        best_fitness = gen_best_fit
        best_solution = gen_best

# --- نمایش نتایج نهایی ---
school_counts = [0] * num_schools  # شمارنده خانه‌های اختصاص یافته به هر مدرسه
for school in best_solution:
    school_counts[school] += 1

print("Best total cost (fitness):", best_fitness)  # نمایش مجموع فاصله نهایی
print("School assignment counts:", school_counts)  # نمایش تعداد خانه‌ها برای هر مدرسه
print("First 20 assignments:", best_solution[:20])  # نمایش ۲۰ تخصیص اول


Best total cost (fitness): 307407.41768375377
School assignment counts: [38, 38, 38, 38, 38, 38, 38, 38, 38, 38]
First 20 assignments: [9, 3, 0, 5, 2, 0, 4, 4, 5, 5, 5, 1, 6, 2, 1, 2, 5, 5, 4, 8]
